In [1]:
%matplotlib inline
# Cell 1 — WO-02 Step 3, family 4: Coverage and shortfall. Load fixtures and population payloads (read-only; the API is never called)
import sys
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.edop.dashboard import wo02_payloads as P, wo02_rules_coverage as V, wo02_analysis as A

fx, pop, prov = P.load_all()
W = P.population_weights(pop)
FX_BY_ID = {q['id']: q for q in fx}
POP_BY_ID = {q['id']: q for q in pop}
print('provenance:', prov)
print(f'fixtures with a 200: {len(fx)} | population queries with a 200: {len(pop)}')


provenance: {'app_version': '0.4', 'git_commit': 'bb80fee', 'git_describe': 'v0.4.2-12-gbb80fee', 'branch': 'dashboard', 'planned': '2026-10-30', 'captured': '2026-10-30'}
fixtures with a 200: 33 | population queries with a 200: 505


In [2]:
# Cell 2 — The four Coverage-and-shortfall statement types (rules in scripts/edop/dashboard/wo02_rules_coverage.py)
for s in V.STATEMENTS:
    print(f"{s['id']}  {s['name']}")
    print(f"      applies to: {s['applies_to']}")
    print(f"      template  : {s['template']}")
print()
print("COV-01, 02 and 04 are about the query (one record per query); COV-03 is about a row (one record per Band A-E row)")
print(f"COV-01 display floor: {V.FLOOR}")


COV-01  Share of the area outside any basin (shortfall)
      applies to: buffer, polity; L6, L8
      template  : {share} of this {area} lies outside any basin (open water, or ground the basin data do not cover); the figures describe the rest.
COV-02  How many basins the figures describe
      applies to: buffer, polity; L6, L8
      template  : The figures for this {area} describe {n} basins (level {level}). | This {area} contains one basin, so the values are that basin's own.
COV-03  A value built from only some of the basins
      applies to: buffer, polity; area_weighted and class_mixture; L6, L8
      template  : {name} has a value in {k} of the {N} basins, which make up {share} of the basin area; the figure rests on those basins.
COV-04  Edge basins of a polity (marginal exposure)
      applies to: polity; L6, L8
      template  : Basins less than half inside this polity's boundary carry {share} of the weight in these figures, and basins less than a fifth inside carry {share}.



In [3]:
# Cell 3 — Fixtures: apply every rule; count where each applies and why not
fdf = A.apply_fixtures(fx, V)
print(A.fixture_table(fdf).to_string())
print()
print('why a statement did not apply:')
print(A.reason_table(fdf).to_string())


res                  applied  not_applicable
id     scope                                
COV-01 buffer              2               3
       polity              5               0
       single_basin        0              23
COV-02 buffer              5               0
       polity              5               0
       single_basin        0              23
COV-03 buffer             13             247
       polity             14             194
       single_basin        0            1196
COV-04 buffer              0               5
       polity              5               0
       single_basin        0              23

why a statement did not apply:
id      reason                      
COV-01  below_floor                        3
        single_basin                      23
COV-02  single_basin                      23
COV-03  all_basins                       369
        method:dominant_basin             27
        single_basin                    1196
        status:low_coverage    

In [4]:
# Cell 4 — Draft sentences on chosen queries (up to 2 per statement type), to read and judge
for qid in ['F03', 'F04', 'F18', 'F19', 'F20', 'F22']:
    g = fdf[(fdf.qid == qid) & fdf.applies]
    print(f"--- {qid}: {FX_BY_ID[qid]['condition']}")
    for sid, gg in g.groupby('id'):
        for _, r in gg.head(2).iterrows():
            print(f"  {sid}  {r['text']}")
    print()
print('--- P069b (population): a 100 km buffer at Shetland')
for r in V.evaluate_query(POP_BY_ID['P069b']):
    if r['applies'] and r['id'] in ('COV-01', 'COV-02'):
        print(f"  {r['id']}  {r['text']}")


--- F03: small buffer (fewer than 5 basins)
  COV-02  The figures for this buffer describe 4 basins (level 6).

--- F04: large buffer; suppressed scores; section 2.6 reference counts
  COV-02  The figures for this buffer describe 9 basins (level 6).
  COV-03  Clay content has a value in 8 of the 9 basins, which make up 84% of the basin area; the figure rests on those basins.
  COV-03  Clay content (upstream) has a value in 8 of the 9 basins, which make up 84% of the basin area; the figure rests on those basins.

--- F18: low_coverage by sea (about 50% sea)
  COV-01  50% of this buffer lies outside any basin (open water, or ground the basin data do not cover); the figures describe the rest.
  COV-02  The figures for this buffer describe 11 basins (level 6).

--- F19: Caspian shore: low_coverage, shortfall about 0.75
  COV-01  75% of this buffer lies outside any basin (open water, or ground the basin data do not cover); the figures describe the rest.
  COV-02  The figures for this buffer

In [5]:
# Cell 5 — Population sample: how often each statement applies. COV-01/02/04 count queries; COV-03 counts Band A-E rows
pdf = A.apply_population(pop, W, V)
print('WEIGHTED by climate-zone size (percent of items):')
print((A.rates(pdf) * 100).round(1).to_string())
print()
print('UNWEIGHTED (percent of items):')
print((A.unweighted_rates(pdf) * 100).round(1).to_string())
print()
print('polities are not in the population sample (5 queries, all fixtures); see Cell 3')


WEIGHTED by climate-zone size (percent of items):
kind    buffer L6  single L6  single L8
id                                     
COV-01       18.9        0.0        0.0
COV-02      100.0        0.0        0.0
COV-03        1.3        0.0        0.0
COV-04        0.0        0.0        0.0

UNWEIGHTED (percent of items):
kind    buffer L6  single L6  single L8
id                                     
COV-01       23.2        0.0        0.0
COV-02      100.0        0.0        0.0
COV-03        2.7        0.0        0.0
COV-04        0.0        0.0        0.0

polities are not in the population sample (5 queries, all fixtures); see Cell 3


In [10]:
# Cell 6 — The numbers inside the statements (weighted)
b = pdf[(pdf.id == 'COV-01') & (pdf.kind == 'buffer L6')].copy()
b['sf'] = b.stat.astype(float)
print('shortfall of the 100 km buffers (share of the circle outside any basin), weighted quantiles:')
print({f'p{int(q*100)}': round(P.wquantile(b.sf, b.w, q), 3) for q in (0.1, 0.25, 0.5, 0.75, 0.9)})
print()
n = pdf[(pdf.id == 'COV-02') & (pdf.kind == 'buffer L6')].copy()
n['k'] = n.stat.astype(float)
bins = {'1': n.k == 1, '2-4': (n.k >= 2) & (n.k <= 4), '5-9': (n.k >= 5) & (n.k <= 9), '10-19': (n.k >= 10) & (n.k <= 19), '20+': n.k >= 20}
print('number of basins in a 100 km buffer, weighted share of buffers (percent):')
print({k: round(float((n.w * m).sum() / n.w.sum() * 100), 1) for k, m in bins.items()})
print('range:', int(n.k.min()), 'to', int(n.k.max()), '| weighted median:', P.wquantile(n.k, n.w, 0.5))
print()
print('COV-03: share of the area with data that carries a value, weighted quantiles:')
print(A.stat_quantiles(pdf, ['COV-03']).round(3).to_string(index=False))
c3 = pdf[(pdf.id == 'COV-03') & pdf.applies]
print()
print('COV-03 rows by variable (count):')
print(c3.variable.value_counts().head(12).to_string())


shortfall of the 100 km buffers (share of the circle outside any basin), weighted quantiles:
{'p10': 0.0, 'p25': 0.0, 'p50': 0.0, 'p75': 0.001, 'p90': 0.115}

number of basins in a 100 km buffer, weighted share of buffers (percent):
{'1': 0.3, '2-4': 6.9, '5-9': 39.9, '10-19': 52.9, '20+': 0.0}
range: 1 to 17 | weighted median: 10.0

COV-03: share of the area with data that carries a value, weighted quantiles:
    id      kind   n   p10   p25   p50   p75   p90
COV-03 buffer L6 140 0.488 0.763 0.899 0.943 0.975

COV-03 rows by variable (count):
variable
wetland_class        32
pct_clay_upstream    19
pct_sand_upstream    19
pct_silt_upstream    19
pct_clay             17
pct_sand             17
pct_silt             17


In [11]:
# Cell 7 — Sweep: the COV-01 display floor. How many buffers would get a shortfall sentence at each floor?
sf_all = pd.DataFrame([(q['id'], q['body']['shortfall']) for q in pop if q['body']['scope']['type'] == 'buffer'], columns=['qid', 'sf'])
sf_all['w'] = sf_all.qid.map(W)
FLOORS = [0.005, 0.01, 0.05, 0.10, 0.25, 0.50]
print('weighted share of 100 km buffers whose shortfall is at least each floor:')
print(pd.DataFrame({f'>= {f}': [float((sf_all.w * (sf_all.sf >= f)).sum() / sf_all.w.sum())] for f in FLOORS}).round(3).to_string(index=False))
print('unweighted:', {f'>= {f}': round(float((sf_all.sf >= f).mean()), 3) for f in FLOORS})
print()
print('polities (5 fixtures), shortfall:', {q['id']: round(q['body']['shortfall'], 4) for q in fx if q['body']['scope']['type'] == 'polity'})


weighted share of 100 km buffers whose shortfall is at least each floor:
 >= 0.005  >= 0.01  >= 0.05  >= 0.1  >= 0.25  >= 0.5
    0.214    0.189    0.183   0.129    0.078   0.008
unweighted: {'>= 0.005': 0.242, '>= 0.01': 0.232, '>= 0.05': 0.192, '>= 0.1': 0.162, '>= 0.25': 0.091, '>= 0.5': 0.02}

polities (5 fixtures), shortfall: {'F22': 0.0109, 'F23': 0.0187, 'F24': 0.019, 'F25': 0.0105, 'F29': 0.0109}


In [12]:
# Cell 8 — Checks: how coverage relates to shortfall, and continuous rows that read ok while resting on little of the area
rel = V.coverage_relation(pop + fx)
print('row coverage versus 1 - shortfall, by scope and method (equal / below / above):')
for k, v in sorted(rel.items()):
    print('  ', k, v)
print()
sp = pd.DataFrame(V.silent_partial(pop + fx), columns=['qid', 'scope', 'variable', 'coverage', 'share'])
print(f'continuous buffer/polity rows with status ok but a value in only some basins: {len(sp)}')
print('  with coverage below 0.5 (where a categorical row would read low coverage):', int((sp.coverage < 0.5).sum()))
print('  coverage quantiles:', {f'p{int(q*100)}': round(float(sp.coverage.quantile(q)), 3) for q in (0.1, 0.25, 0.5, 0.75, 0.9)})
print('  variables:')
print(sp.variable.value_counts().head(10).to_string())
print()
d = [(r['qid'], r['variable']) for q in pop + fx for r in V.evaluate_query(q) if r['id'] == 'COV-03' and r['applies']]
print('COV-03 rows (applied):', len(d), '| continuous ok partial rows found independently:', len(sp), '| the rest are categorical rows')


row coverage versus 1 - shortfall, by scope and method (equal / below / above):
   ('buffer', 'area_weighted', 'below') 228
   ('buffer', 'area_weighted', 'equal') 3412
   ('buffer', 'class_mixture', 'below') 60
   ('buffer', 'class_mixture', 'equal') 980
   ('buffer', 'distribution_only', 'equal') 312
   ('buffer', 'dominant_basin', 'above') 99
   ('buffer', 'dominant_basin', 'equal') 213
   ('buffer', 'flag_fraction', 'equal') 104
   ('polity', 'area_weighted', 'below') 9
   ('polity', 'area_weighted', 'equal') 131
   ('polity', 'class_mixture', 'below') 4
   ('polity', 'class_mixture', 'equal') 36
   ('polity', 'distribution_only', 'equal') 12
   ('polity', 'dominant_basin', 'above') 12
   ('polity', 'flag_fraction', 'equal') 4
   ('single_basin', 'area_weighted', 'below') 702
   ('single_basin', 'area_weighted', 'equal') 14313
   ('single_basin', 'class_mixture', 'below') 136
   ('single_basin', 'class_mixture', 'equal') 4154
   ('single_basin', 'distribution_only', 'equal') 1287
 

In [6]:
# Cell 9 — Write this family's rows to the master table (replaces any earlier Coverage and shortfall rows)
fxt, rw, ru = A.fixture_table(fdf), A.rates(pdf), A.unweighted_rates(pdf)
rows = []
for s in V.STATEMENTS:
    sid = s['id']
    f = fxt.loc[sid]
    unit = 'rows' if sid == 'COV-03' else 'queries'
    row = dict(s, family='Coverage and shortfall')
    row['fixture_result'] = f"{int(f.applied.sum())} {unit} applied, {int(f.not_applicable.sum())} not applicable ({len(fx)} fixtures)"
    row['population_result'] = (f'share of {unit}, weighted: ' + ', '.join(f"{k} {rw.loc[sid, k]:.1%}" for k in rw.columns)
                                + '; unweighted: ' + ', '.join(f"{k} {ru.loc[sid, k]:.1%}" for k in ru.columns) + '; polities not sampled')
    row['sweep_result'] = ('display floor swept 0.5% to 50% (Cell 7)' if sid == 'COV-01' else 'no cutoff')
    rows.append(row)
n_rows = P.upsert_master('Coverage and shortfall', rows)
print('master table now has', n_rows, 'rows ->', P.MASTER)


master table now has 22 rows -> /Users/karlg/Documents/repos/_edops/notebooks/edop/dashboard/wo02_master_table.csv
